# 06 — RUN PHASE 2: PCC vs baseline, dua tabel, banyak skenario

Runner **tipis** di atas `pcc/experiments/phase2_pcc.py`. Semua logikanya ada di
skrip itu — notebook ini hanya menyiapkan data, membangun grid, dan mengagregasi.
Itu yang diminta `AGENTS.md` §4/§12: skrip ber-`--seed` yang reproducible bit-per-bit
adalah jalur eksekusi utama, notebook hanya pembungkusnya.

**Sel penyiapan data di bawah disalin apa adanya dari notebook 05.** Logika itu
menghabiskan lima bug untuk dibetulkan (kuota gdown, baca header `.npz`, materialisasi
selektif, pemeriksaan skip, konvensi penamaan), jadi ia dipakai ulang byte-per-byte,
bukan ditulis ulang.

Yang dihasilkan: satu laporan JSON per konfigurasi di `pcc/reports/`, plus ringkasan
teragregasi antar-seed di akhir.

| Yang diuji | Di mana klaimnya |
|---|---|
| **Tabel 1** kelas ber-data | PCC harus **minimal seri** dengan Clustered CP |
| **Tabel 2** kelas `n_y = 0` | tempat klaim ekstrapolasi hidup |

Menang di Tabel 2 saja **bukan** lulus: di sana setiap pesaing memang tak terdefinisi.
Kriteria lengkap ada di `reports/baseline_reproduction.md` dan dieksekusi oleh driver.

In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = ''
REPO_DIR   = 'foundation-cp'
DRIVE_ROOT = '/content/drive/MyDrive/pcc'

# SUMBER DUMP. LTC sudah punya ID gdown konkret (dari notebook 00) -> nol langkah
# manual. CCC belum: ID-nya harus dibaca dari download_data.sh mereka. Jadi survei
# LTC dulu; pindah ke CCC hanya kalau tidak ada dump LTC yang memenuhi premis.
# TIDAK ADA saklar sumber. Versi sebelumnya punya SOURCE yang harus diedit manual,
# dan default-nya ('ltc') sudah diketahui GAGAL premis — jadi setiap run default
# berakhir dengan assert. Sekarang SEMUA sumber disurvei dalam satu jalan dan yang
# terbaik dipakai. Tabel perbandingannya sendiri adalah temuan yang dicari.
CCC_DATASETS = ('imagenet',)   # tambah 'inaturalist' (iNat-2021, 633 kelas) bila perlu
GID_SCORES_LTC = {'plantnet': '1k_PPQV3VJT44hz02CcnbqPstjQo70vGr',
                  'inaturalist': '1W8R8Jj2bhS2PbR-3X9vEw-WkanbOk6mq'}
# ID CCC dari download_data.sh mereka. Ditanam supaya TIDAK ada langkah manual:
# versi sebelumnya hanya mencetak skripnya dan menyuruh menjalankan gdown sendiri,
# yang membingungkan dan tidak perlu begitu ID-nya diketahui.
GID_SCORES_CCC = {'imagenet':   '1AQjUn3m010N_i6-sfD690W7mZq2RTwJz',   # 4,62 GB
                  'inaturalist':'1BUlQZhS_5x2LJpyxCGI1IkmRrkvmRD88',   # 6,72 GB (iNat-2021, 633 kelas)
                  'places365':  '119k7PE1l72fg5Rpez5brIOn28BwClqv2',   # 0,54 GB (365 kelas)
                  'cifar-100':  '1yXD9XqBxEJnJxcfnnduNK6nHHUU3iX_6'}   # 0,01 GB (100 kelas)
# Catatan daya uji: premis butuh >=500 kelas layak, jadi places365 (365 kelas) dan
# cifar-100 (100 kelas) TIDAK BISA memenuhinya secara konstruksi, berapa pun
# sampel per kelasnya. Yang mungkin: imagenet (1.000) dan iNat-2021 (633).
LTC_DATASETS = ('inaturalist', 'plantnet')   # rilis memuat varian -trunc juga
LOSS_VARIANT = 'cross_entropy'  # 'cross_entropy' | 'focal'. LTC mengirim SEMBILAN
                                # berkas dengan NAMA IDENTIK di subdirektori berbeda;
                                # tercampur = skor dari model lain, akurasi mirip,
                                # kalibrasi beda total. Notebook 00 kena isu yang sama.
# places365 (365 kelas) dan cifar-100 (100 kelas) TIDAK BISA memenuhi premis >=500
# kelas secara konstruksi, berapa pun sampel per kelasnya — jadi tidak diunduh.
N_CLASSES_EXPECTED = None      # None = jangan dipaksakan; dibaca dari dump

# --- PRIMER, ditetapkan di prereg_imagenet_gate.md. JANGAN diubah setelah melihat hasil.
ALPHA_PRIMARY = 0.10
N_CAL_PRIMARY = 25
N_BOOT_CLASS  = 400           # bootstrap tingkat-kelas untuk gate B
N_PERM_CLASS  = 1000          # permutasi tingkat-kelas untuk gate C (p_min = 1/1001)
STABLE_THRESHOLD = 0.90

# --- SEKUNDER
ALPHAS_SECONDARY = (0.01, 0.05)
N_CAL_SECONDARY  = 50
N_SPLITS_BC = 100
N_SPLITS_A  = 100
RUN_CLUSTERED_CP = True       # reproduksi baseline pada skor yang sama

FRAC_DESC, FRAC_CAL = 0.40, 0.30   # sisanya EVAL

# ANGGARAN BARIS. Dump ImageNet CCC nyata adalah (1.153.051 x 1.000) float32 = 4,61 GB
# -- sepuluh kali lebih besar dari yang tercatat di release_audit.md. Memuatnya penuh
# lalu membuat salinan turunan (thr_lac, entropi, np.partition) melewati RAM Colab.
#
# Subsampling di sini BUKAN perubahan kriteria: premis butuh >=84 sampel/kelas dan
# anggaran ini menyisakan ~230/kelas. Ia diambil sebagai FRAKSI per kelas, bukan cap
# tetap, karena cap tetap membuat semua hitungan kelas SAMA -> log_prevalence konstan
# -> ablasi prevalensi jadi hampa. Fraksi mempertahankan struktur prevalensinya.
MAX_ROWS = 250_000            # None = pakai seluruh dump
SEED = 42
# =======================================================================
print(f'PRIMER: alpha={ALPHA_PRIMARY} n_cal={N_CAL_PRIMARY} '
      f'n_boot={N_BOOT_CLASS} n_perm={N_PERM_CLASS}')


## 2. Mount + repo + env


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR if os.path.isabs(REPO_DIR) else '/content/'+REPO_DIR)
os.environ['PYTHONPATH'] = os.getcwd() + os.pathsep + os.environ.get('PYTHONPATH','')
subprocess.run(['pip','install','-q','-r','requirements.txt'], check=False)
subprocess.run(['pip','install','-q','gdown'], check=False)
from pcc.utils.seed import set_seed; from pcc.utils.io import environment_stamp
set_seed(SEED)
print('env:', environment_stamp()['packages'])


## 3. Siapkan SEMUA dump — TANPA citra, TANPA GPU

Dump LTC dipakai lokasi yang sama dengan notebook 00 (`released_scores/<dataset>`), jadi kalau
sudah ada tidak diunduh ulang. Dump CCC diunduh otomatis dengan ID yang sudah ditanam.

Keduanya disurvei berdampingan di sel 4. Itu bukan pemborosan: **perbandingan ekor-panjang
versus berimbang adalah temuannya**, dan menurunkannya dari satu tabel lebih kuat daripada dari
dua run terpisah.


In [ ]:
import glob, zipfile, numpy as np

def ltc_dir(ds):
    return f'{DRIVE_ROOT}/released_scores/{ds}'

def ccc_dir(ds):
    return f'{DRIVE_ROOT}/scores_ccc/{ds}'

for ds in LTC_DATASETS:
    d = ltc_dir(ds); os.makedirs(d, exist_ok=True)
    if glob.glob(f'{d}/**/*_softmax.npy', recursive=True):
        print(f'ltc/{ds}: sudah ada, dilewati')
        continue
    print(f'ltc/{ds}: mengunduh...')
    z = f'{d}/{ds}.zip'
    r = subprocess.run(['gdown', GID_SCORES_LTC[ds], '-O', z], capture_output=True, text=True)
    if r.returncode:
        print('  gdown gagal:', r.stderr.strip()[:300])
    else:
        subprocess.run(['unzip','-o','-q',z,'-d',d], check=False)

def inventory(d, label):
    files = [p for p in sorted(glob.glob(f'{d}/**/*', recursive=True)) if os.path.isfile(p)]
    print(f'  isi {label}: {len(files)} berkas')
    for p in files[:25]:
        print(f'    {os.path.relpath(p, d):56s} {os.path.getsize(p)/1e6:9.2f} MB')
    return files

def looks_like_html(p):
    # Kegagalan kuota Google Drive menulis halaman HTML DAN mengembalikan kode 0.
    # Inilah sebabnya returncode tidak boleh dipercaya sebagai bukti unduhan berhasil.
    try:
        with open(p, 'rb') as fh:
            head = fh.read(400)
    except OSError:
        return False, b''
    low = head.lower()
    return (b'<html' in low or b'<!doctype html' in low), head

for ds in CCC_DATASETS:
    d = ccc_dir(ds); os.makedirs(d, exist_ok=True)
    mat = f'/content/ccc_npy/{ds}'
    # Tiga keadaan, dan versi sebelumnya hanya mengenali yang pertama:
    #   (a) .npy sudah dimaterialkan di /content -> tidak ada kerja
    #   (b) .npz ada di Drive tapi .npy hilang (sesi baru; /content ephemeral)
    #       -> ekstrak ulang, JANGAN unduh 4,6 GB lagi
    #   (c) tidak ada apa pun -> unduh
    # Pemeriksaan lama hanya mencari .npy DI DRIVE, yang tidak pernah ada karena
    # materialisasinya ke /content. Jadi setiap sesi baru mengunduh ulang 4,6 GB.
    if glob.glob(f'{mat}/*.npy') or glob.glob(f'{d}/**/*.npy', recursive=True):
        print(f'ccc/{ds}: .npy sudah ada, dilewati')
        continue
    if glob.glob(f'{d}/*.npz'):
        print(f'ccc/{ds}: .npz ada di Drive, ekstrak ulang tanpa mengunduh')
    else:
        print(f'ccc/{ds}: mengunduh (beberapa GB, sabar)...')
    if not glob.glob(f'{d}/*.npz'):
        r = subprocess.run(['gdown', '--fuzzy', GID_SCORES_CCC[ds]],
                           cwd=d, capture_output=True, text=True)
        if r.stdout.strip():
            print('  stdout:', r.stdout.strip()[-500:])
        if r.stderr.strip():
            print('  stderr:', r.stderr.strip()[-300:])
        print(f'  returncode: {r.returncode}  <- BUKAN bukti; diverifikasi di bawah')

    for p in sorted(glob.glob(f'{d}/*')):
        low = p.lower()
        if low.endswith('.zip'):
            subprocess.run(['unzip','-o','-q',p,'-d',d], check=False)
        elif low.endswith(('.tar.gz','.tgz','.tar')):
            subprocess.run(['tar','-xf',p,'-C',d], check=False)

    # .npz ADALAH zip berisi beberapa .npy. Versi sebelumnya mencocokkan ekstensi
    # secara literal ('.zip'/'.tar'), jadi imagenet.npz dilewati dan tidak ada .npy
    # terbentuk -- padahal unduhannya berhasil penuh. Satu baris, kegagalan bisu.
    #
    # Header .npy dibaca lewat zipfile TANPA mendekompresi isinya, supaya bentuk dan
    # dtype tiap anggota terlihat tanpa memuat gigabyte. Lalu HANYA dua array yang
    # dibutuhkan dimaterialkan, dan ke /content (ephemeral) bukan Drive -- mengekstrak
    # seluruh 4,6 GB ke Drive akan menggandakan pemakaian kuota tanpa alasan.
    for p in sorted(glob.glob(f'{d}/*.npz')):
        print(f'  membaca header {os.path.basename(p)} (tanpa dekompresi)...')
        members = []
        with zipfile.ZipFile(p) as zf:
            for nm in zf.namelist():
                try:
                    with zf.open(nm) as fh:
                        ver = np.lib.format.read_magic(fh)
                        if ver == (1, 0):
                            shp, _fo, dt = np.lib.format.read_array_header_1_0(fh)
                        elif ver == (2, 0):
                            shp, _fo, dt = np.lib.format.read_array_header_2_0(fh)
                        else:
                            continue
                    members.append((nm, shp, dt))
                    print(f'    {nm:36s} {str(shp):20s} {dt}')
                except Exception as e:
                    print(f'    {nm:36s} header tak terbaca: {type(e).__name__}')

        twod = [m for m in members if len(m[1]) == 2]
        oned = [m for m in members if len(m[1]) == 1]
        if not twod or not oned:
            print('  npz ini tidak memuat pasangan (2-D, 1-D) — kirim daftar di atas.')
            continue
        sc = max(twod, key=lambda m: m[1][0] * m[1][1])
        lb = next((m for m in oned if m[1][0] == sc[1][0]), None)
        if lb is None:
            print(f'  tidak ada array 1-D sepanjang {sc[1][0]} untuk mendampingi {sc[0]}')
            continue
        out = f'/content/ccc_npy/{ds}'
        os.makedirs(out, exist_ok=True)
        print(f'  materialkan {sc[0]} {sc[1]} dan {lb[0]} {lb[1]} -> {out}')
        # zipfile.namelist() memberi nama DENGAN sufiks '.npy', tetapi NpzFile
        # diindeks TANPA sufiks -> z['softmax.npy'] KeyError, z['softmax'] benar.
        # Ditemukan oleh tes sintetik sebelum run nyata.
        k_sc = sc[0][:-4] if sc[0].endswith('.npy') else sc[0]
        k_lb = lb[0][:-4] if lb[0].endswith('.npy') else lb[0]
        with np.load(p, allow_pickle=False) as z:
            np.save(f'{out}/scores.npy', z[k_sc])
            np.save(f'{out}/labels.npy', z[k_lb])

    files = inventory(d, f'ccc/{ds}')
    npys = (glob.glob(f'{d}/**/*.npy', recursive=True)
            + glob.glob(f'/content/ccc_npy/{ds}/*.npy'))
    if not npys:
        print(f'  GAGAL: tidak ada .npy terbentuk untuk ccc/{ds}.')
        for p in files:
            is_html, head = looks_like_html(p)
            if is_html:
                print(f'  PENYEBAB: {os.path.basename(p)} adalah HALAMAN HTML, bukan data.')
                print('  Itu batas kuota Google Drive; gdown tetap keluar dengan kode 0.')
                print('  cuplikan:', ' '.join(head.decode('utf-8','replace').split())[:300])
                print(f'  URL: https://drive.google.com/uc?id={GID_SCORES_CCC[ds]}')
                break
        else:
            print('  Berkas ADA tetapi tidak menghasilkan .npy — kirim daftar di atas.')
    else:
        print(f'  OK: {len(npys)} .npy siap dipakai')

# dua akar untuk CCC: Drive (kalau .npy langsung) dan /content (hasil materialisasi
# anggota .npz). Keduanya diperiksa supaya tidak peduli bentuk rilisnya.
roots = ([(ltc_dir(ds), 'ltc', ds) for ds in LTC_DATASETS]
         + [(ccc_dir(ds), 'ccc', ds) for ds in CCC_DATASETS]
         + [(f'/content/ccc_npy/{ds}', 'ccc', ds) for ds in CCC_DATASETS])
found = []
for rt, src, ds in roots:
    for f in sorted(glob.glob(f'{rt}/**/*.npy', recursive=True)):
        found.append((f, src, ds))
print()
print(f'total .npy: {len(found)}')
for f, src, ds in found[:60]:
    a = np.load(f, mmap_mode='r')
    print(f'  [{src}] {os.path.basename(f):52s} {str(a.shape):18s} {a.dtype}')
if not found:
    print('TIDAK ADA .npy sama sekali. Isi direktori mentah:')
    for rt, src, ds in roots:
        for p in sorted(glob.glob(f'{rt}/**/*', recursive=True))[:25]:
            if os.path.isfile(p):
                print(f'  {os.path.relpath(p, DRIVE_ROOT):64s} {os.path.getsize(p)/1e6:8.1f} MB')


### 3b. Referensi — dari mana ID CCC berasal (opsional, tidak perlu dijalankan)

ID di sel 1 diambil dari `download_data.sh` milik CCC. Sel ini hanya untuk memverifikasi
bahwa ID-nya belum berubah; ia **tidak diperlukan** untuk menjalankan notebook.


In [ ]:
SHOW_CCC_SCRIPT = False
if SHOW_CCC_SCRIPT:
    if not os.path.isdir('/content/ccc'):
        subprocess.run(['git','clone','--depth','1',
                        'https://github.com/tiffanyding/class-conditional-conformal.git',
                        '/content/ccc'], check=False)
    print(open('/content/ccc/download_data.sh').read())
    print('bandingkan dengan GID_SCORES_CCC di sel 1')
else:
    print('dilewati (ID sudah ditanam di sel 1)')


## 6. Bobot kepala klasifier → `.npy`

Driver menerima `--head-weights` sebagai berkas, jadi kepala torchvision diunduh sekali
(~100 MB, tanpa GPU) lalu disimpan. Ia model **tersupervisi yang berbeda** dari
SimCLRv2+probe yang menghasilkan skor CCC — ketidakcocokan itu justru intinya, karena
itulah yang membuat φ eksogen terhadap δ_y.

In [ ]:
import numpy as np, os

HEAD_DIR = '/content/head'
os.makedirs(HEAD_DIR, exist_ok=True)
HEAD_W = f'{HEAD_DIR}/resnet50_fc_weight.npy'
HEAD_B = f'{HEAD_DIR}/resnet50_fc_bias.npy'

if not (os.path.exists(HEAD_W) and os.path.exists(HEAD_B)):
    from pcc.descriptors.head_weights import load_torchvision_resnet50_head
    W, b = load_torchvision_resnet50_head()
    np.save(HEAD_W, W)
    np.save(HEAD_B, np.zeros(len(W)) if b is None else b)
    print('kepala disimpan:', W.shape)
else:
    print('kepala sudah ada, dilewati')
K_HEAD = int(np.load(HEAD_W, mmap_mode='r').shape[0])
print('kelas pada kepala:', K_HEAD, '-> hanya cocok untuk dump ImageNet-1k')

## 7. Temukan dump yang siap dipakai

Jalur dicari, bukan diasumsikan, dan dicetak sebelum dipakai. LTC memisahkan dump
kalibrasi dan test, jadi keduanya dipasangkan: DESC+CAL dari dump cal, EVAL dari
**seluruh** dump test. Kalau dump test dibelah 40/30/30, Pl@ntNet hanya menyisakan ~1
baris evaluasi per kelas — di bawah ambang rezim B yang sudah dipra-registrasi di
`reports/prereg_metrics_per_dataset.md`.

In [ ]:
import glob

def _pair(scores):
    for suf in ('_softmax.npy', '_scores.npy', 'scores.npy'):
        if scores.endswith(suf):
            cand = scores[: -len(suf)] + suf.replace('softmax', 'labels').replace(
                'scores', 'labels')
            if os.path.exists(cand):
                return cand
    cand = os.path.join(os.path.dirname(scores), 'labels.npy')
    return cand if os.path.exists(cand) else None

DUMPS = {}

# CCC: materialisasi sel di atas menaruhnya di /content/ccc_npy/<ds>/
for p in sorted(glob.glob('/content/ccc_npy/*/scores.npy')):
    ds = os.path.basename(os.path.dirname(p))
    lab = _pair(p)
    if lab:
        DUMPS['ccc_' + ds] = {'scores': p, 'labels': lab, 'eval_scores': None,
                              'eval_labels': None, 'max_rows': MAX_ROWS}

# LTC: pasangkan cal (DESC+CAL) dengan test (EVAL penuh)
for ds in LTC_DATASETS:
    d = f'{DRIVE_ROOT}/released_scores/{ds}'
    cal = sorted(glob.glob(f'{d}/**/*cal_softmax.npy', recursive=True))
    tst = sorted(glob.glob(f'{d}/**/*test_softmax.npy', recursive=True))
    cal = [p for p in cal if LOSS_VARIANT in p or LOSS_VARIANT == 'cross_entropy']
    tst = [p for p in tst if LOSS_VARIANT in p or LOSS_VARIANT == 'cross_entropy']
    if cal and tst and _pair(cal[0]) and _pair(tst[0]):
        DUMPS['ltc_' + ds] = {'scores': cal[0], 'labels': _pair(cal[0]),
                              'eval_scores': tst[0], 'eval_labels': _pair(tst[0]),
                              'max_rows': None}

assert DUMPS, 'tidak ada dump siap pakai -- periksa sel penyiapan di atas'
for k, v in DUMPS.items():
    a = np.load(v['scores'], mmap_mode='r')
    line = '  ' + k.ljust(18) + ' cal ' + str(a.shape)
    if v['eval_scores']:
        e = np.load(v['eval_scores'], mmap_mode='r')
        line += '  eval ' + str(e.shape) + '  (dump terpisah)'
    else:
        line += '  eval = 30% dari dump yang sama'
    print(line)

## 8. Grid

Satu tempat untuk semua sumbu. `phi='head'` hanya sah bila jumlah kelas dump sama
dengan kepala (1000), jadi keluarga φ dipilih per dataset, bukan disamakan buta.

Seed dijalankan penuh hanya pada sel headline; sumbu lain memakai satu seed supaya
grid tidak meledak. Yang dipangkas **dicetak**, bukan dihilangkan diam-diam.

In [ ]:
SEEDS_HEADLINE = (0, 1, 2, 3, 4)
SEED_SWEEP     = (0,)
ALPHAS         = (0.10, 0.05)
N_CALS_WANTED  = (25, 50)
HELDOUT_FRACS  = (0.30,)
CCC_ROOT       = '/content/ccc' if os.path.isdir('/content/ccc') else None

def phi_families(K):
    fams = ['output']
    if K == K_HEAD:
        fams.insert(0, 'head')      # keluarga non-sirkular lebih dulu
    return fams

def n_cals_for(v, K):
    # n_cal must be REACHABLE. Pl@ntNet's released calibration dump has a median
    # of 2 rows per class, so n_cal=25 is impossible there by construction and the
    # driver would (correctly) refuse. Derive what the data can support instead of
    # forcing a number, and print what was dropped -- silently lowering a
    # pre-registered criterion is exactly what must not happen.
    y = np.load(v['labels'])
    cnt = np.bincount(y, minlength=K)
    frac = (1.0 - FRAC_DESC) if v['eval_scores'] else FRAC_CAL
    per = np.sort(cnt * frac)[::-1]
    need = 30                      # want at least ~30 trainable classes for g_theta
    cap = int(per[min(need, len(per)) - 1]) if len(per) else 0
    ok = [nc for nc in N_CALS_WANTED if nc <= cap]
    if ok:
        return ok, cap, []
    fallback = max(5, min(cap, 10))
    return ([fallback] if fallback <= cap else []), cap, list(N_CALS_WANTED)

GRID, NCAL_NOTES = [], {}
for name, v in DUMPS.items():
    K = int(np.load(v['scores'], mmap_mode='r').shape[1])
    ncals, cap, dropped = n_cals_for(v, K)
    NCAL_NOTES[name] = {'used': ncals, 'reachable_cap': cap, 'dropped': dropped}
    if not ncals:
        print('DILEWATI', name, '- bahkan n_cal=5 tak tercapai (cap', cap, ')')
        continue
    for phi in phi_families(K):
        for a in ALPHAS:
            for nc in ncals:
                for hf in HELDOUT_FRACS:
                    headline = (a == ALPHAS[0] and nc == ncals[0])
                    for s in (SEEDS_HEADLINE if headline else SEED_SWEEP):
                        GRID.append(dict(dump=name, K=K, phi=phi, alpha=a,
                                         n_cal=nc, heldout_frac=hf, seed=s,
                                         headline=headline))

print('konfigurasi:', len(GRID))
for name in DUMPS:
    K = int(np.load(DUMPS[name]['scores'], mmap_mode='r').shape[1])
    n = sum(1 for g in GRID if g['dump'] == name)
    nt = NCAL_NOTES[name]
    print('  ' + name.ljust(18), 'K=' + str(K).ljust(6),
          'phi=' + ','.join(phi_families(K)).ljust(12),
          'n_cal=' + str(nt['used']).ljust(10),
          '(cap ' + str(nt['reachable_cap']) + ')', n, 'run')
    if nt['dropped']:
        print('      DIJATUHKAN n_cal', nt['dropped'],
              '- tidak tercapai pada dump ini; dipakai', nt['used'], 'sebagai ganti.')
        print('      Ini BUKAN penurunan diam-diam: dicatat di sini dan di laporan.')
if CCC_ROOT is None:
    print()
    print('CATATAN: repo CCC tidak ada di /content/ccc, jadi baseline Tabel 1')
    print('DILEWATI. Jalankan sel penemuan akar impor di notebook 05 lebih dulu,')
    print('atau clone manual. Ini dicetak supaya ketidakhadirannya tidak terbaca')
    print('sebagai baseline yang gagal.')

## 9. Jalankan

Driver dipanggil dalam proses supaya hasilnya langsung bisa diagregasi; tiap konfigurasi
tetap menulis laporan JSON-nya sendiri lewat `write_report`, jadi jejaknya sama dengan
menjalankan skripnya dari CLI. Kegagalan satu konfigurasi dicatat dan **tidak**
menghentikan sisanya.

In [ ]:
import time, traceback
from pcc.experiments import phase2_pcc as drv

class A:  pass

def build_args(g):
    v = DUMPS[g['dump']]
    a = A()
    a.scores, a.labels = v['scores'], v['labels']
    a.eval_scores, a.eval_labels = v['eval_scores'], v['eval_labels']
    a.max_rows = v['max_rows']
    a.dataset = g['dump']
    a.reports_dir = 'pcc/reports'
    a.alpha, a.n_cal = g['alpha'], g['n_cal']
    a.heldout_frac = g['heldout_frac']
    a.frac_desc, a.frac_cal = FRAC_DESC, FRAC_CAL
    a.phi = g['phi']
    a.head_weights = HEAD_W if g['phi'] == 'head' else None
    a.head_bias = HEAD_B if g['phi'] == 'head' else None
    a.distance_holdout = 'w_cos_knn_1' if g['phi'] == 'head' else 'prof_knn_1'
    a.stat = 'worst'
    a.ccc_root = CCC_ROOT
    a.seed = g['seed']
    a.name = None
    a.print_json = False
    return a

RESULTS, FAILED = [], []
t_start = time.time()
for i, g in enumerate(GRID, 1):
    tag = '{dump}|{phi}|a{alpha}|nc{n_cal}|s{seed}'.format(**g)
    print('[{}/{}] {}'.format(i, len(GRID), tag), flush=True)
    t0 = time.time()
    try:
        a = build_args(g)
        res = drv.run(a)
        concl = drv.verdict(res, a.stat)
        nm = 'phase2_{}_{}_a{}_nc{}_ho{}_s{}'.format(
            g['dump'], g['phi'], g['alpha'], g['n_cal'], g['heldout_frac'], g['seed'])
        drv.write_report(a.reports_dir, nm, hypothesis=drv.HYPOTHESIS,
                         pass_criteria=drv.PASS_CRITERIA, config=vars(a),
                         seed=g['seed'], results=res, conclusion=concl,
                         started_at=t0)
        RESULTS.append(dict(g, res=res, conclusion=concl))
        t1 = res['table_1_seen']
        t2 = res.get('table_2_heldout')
        s1 = t1['primary_stat']
        msg = '    {:.0f}s | lam {:.3f} | n_star {} | T1[{}] {:+.4f}'.format(
            time.time() - t0, res['pcc']['lambda'], res['pcc']['n_star'], s1,
            t1['delta'].get(s1, float('nan')))
        if t2 is not None:
            s2 = t2['primary_stat']
            msg += ' | T2[{}] {:+.4f}'.format(s2, t2['delta'].get(s2, float('nan')))
        print(msg, '|', concl, flush=True)
    except Exception as e:
        FAILED.append(dict(g, error=type(e).__name__ + ': ' + str(e)))
        print('    GAGAL:', FAILED[-1]['error'][:200], flush=True)
        traceback.print_exc()

print()
print('selesai {} / {} dalam {:.0f}s'.format(len(RESULTS), len(GRID),
                                             time.time() - t_start))
if FAILED:
    print('GAGAL {}:'.format(len(FAILED)))
    for f in FAILED:
        print('  {dump}|{phi}|a{alpha}|nc{n_cal}|s{seed}:'.format(**f), f['error'][:140])

## 10. Agregasi antar-seed

CI diambil **antar seed** pada sel headline. Itu bukan CI antar-kelas: unit yang
divariasikan di sini adalah pemisahan data dan pemilihan kelas held-out, jadi yang
diukur adalah stabilitas hasil terhadap pilihan itu — bukan ketidakpastian tingkat
kelas, yang sudah ditangani bootstrap kelas di notebook 05.

In [ ]:
from pcc.eval.stats import mean_ci
from collections import defaultdict

agg = defaultdict(lambda: defaultdict(list))
for r in RESULTS:
    if not r['headline']:
        continue
    key = (r['dump'], r['phi'], r['alpha'], r['n_cal'])
    for tname in ('table_1_seen', 'table_2_heldout'):
        tb = r['res'].get(tname)
        if tb is None:
            continue
        s = tb['primary_stat']
        if s in tb['delta']:
            agg[key][tname + '|' + s].append(tb['delta'][s])
        agg[key][tname + '|macro'].append(tb['delta']['macro'])
        agg[key][tname + '|size_matched'].append(1.0 if tb['size_matched'] else 0.0)
    agg[key]['lambda'].append(r['res']['pcc']['lambda'])

SUMMARY = {}
print('=== HEADLINE, CI antar-seed ===')
for key in sorted(agg):
    print('  ' + '|'.join(str(x) for x in key))
    row = {}
    for metric in sorted(agg[key]):
        vals = np.array(agg[key][metric], float)
        ci = mean_ci(vals)
        row[metric] = ci
        if metric.endswith('size_matched'):
            if ci['mean'] < 1.0:
                print('      {:28s} UKURAN TIDAK COCOK di sebagian seed'.format(metric))
            continue
        flag = ''
        if metric.startswith('table_'):
            flag = '  <- CI di atas 0' if ci['ci_low'] > 0 else ''
        print('      {:28s} {:+.4f} [{:+.4f}, {:+.4f}] n={}{}'.format(
            metric, ci['mean'], ci['ci_low'], ci['ci_high'], ci['n'], flag))
    SUMMARY['|'.join(str(x) for x in key)] = row

## 11. Verdict gabungan dan laporan

Kriteria yang dipatok di `reports/baseline_reproduction.md`: Tabel 2 harus punya CI
antar-seed di atas nol, **dan** Tabel 1 tidak boleh turun lebih dari 0,01. Menang di
Tabel 2 saja bukan lulus — di sana setiap pesaing memang tak terdefinisi.

In [ ]:
import time
from pcc.utils.io import write_report

verdicts = {}
for key, row in SUMMARY.items():
    t2 = [v for k, v in row.items() if k.startswith('table_2') and
          not k.endswith(('macro', 'size_matched'))]
    t1 = [v for k, v in row.items() if k.startswith('table_1') and
          not k.endswith(('macro', 'size_matched'))]
    if not t2 or not t1:
        verdicts[key] = 'TIDAK DAPAT DINILAI'
        continue
    won2 = t2[0]['ci_low'] > 0
    kept1 = t1[0]['mean'] > -0.01
    verdicts[key] = ('LULUS' if won2 and kept1 else
                     'MENUKAR (menang held-out, kalah kelas terlihat)' if won2 else
                     'GAGAL')

for k in sorted(verdicts):
    print('  {:52s} {}'.format(k, verdicts[k]))

overall = ('LULUS' if verdicts and all(v == 'LULUS' for v in verdicts.values())
           else 'SEBAGIAN' if any(v == 'LULUS' for v in verdicts.values())
           else 'GAGAL')
print()
print('GABUNGAN:', overall)

CAVEATS = [
    'Notebook ini runner tipis; semua logika di pcc/experiments/phase2_pcc.py.',
    'CI di sini ANTAR-SEED (pembelahan data + pemilihan kelas held-out), BUKAN',
    '  antar-kelas. Ketidakpastian tingkat kelas ditangani bootstrap kelas di nb 05.',
    'Statistik primer per tabel mengikuti prereg_metrics_per_dataset.md: rezim A',
    '  (worst-class) bila median eval/kelas >= 30, rezim B (bin prevalensi) bila kurang.',
    'Sec 7 BELUM terpenuhi: baseline belum direproduksi lawan angka terbit penulisnya,',
    '  jadi angka baseline sahih untuk perbandingan internal saja.',
]
if CCC_ROOT is None:
    CAVEATS.append('Baseline Tabel 1 DILEWATI: repo CCC tidak ada di /content/ccc.')
for c in CAVEATS:
    print('CAVEAT:', c)

path = write_report('pcc/reports', '06_phase2_summary',
                    hypothesis=drv.HYPOTHESIS, pass_criteria=drv.PASS_CRITERIA,
                    config={'grid_size': len(GRID), 'dumps': sorted(DUMPS),
                            'alphas': list(ALPHAS), 'n_cal_notes': NCAL_NOTES,
                            'heldout_fracs': list(HELDOUT_FRACS),
                            'seeds_headline': list(SEEDS_HEADLINE),
                            'ccc_root': CCC_ROOT, 'seed': SEED},
                    seed=SEED,
                    results={'summary': SUMMARY, 'verdicts': verdicts,
                             'overall': overall, 'n_ok': len(RESULTS),
                             'n_failed': len(FAILED), 'failed': FAILED,
                             'caveats': CAVEATS},
                    conclusion=overall, started_at=t_start)
print()
print('laporan:', path)